# MeetStream Bridge + LangChain — Real, Live Test Notebook

Runs the full bridge + `langchain-meetstream` pipeline described in
[`docs/LANGCHAIN.md`](../docs/LANGCHAIN.md) against the **real** MeetStream API with
**real** credentials — a real bot joins a real meeting, a real transcript is fetched, and
the LangChain loader, tools, and agent all run against that real data. No mocks, no
placeholder bridge key, no skip-if-blank guards.

**Before running**: fill in every `<INSERT ... HERE>` value in section 2 with your real
MeetStream API key and either a real meeting URL or an existing `bot_id`. Cells that need a
value you left blank will raise, not silently skip — that's deliberate, so a half-configured
run fails loudly instead of quietly passing on empty data.

Structure:
1. Install the two local projects (editable, from local source — no PyPI involved)
2. Configuration — insert your real credentials and meeting info here
3. Start a real bridge server, pointed at your real MeetStream key
4. LangChain: client, tools, dispatch, transcript, loader, agent — all real
5. RAG over the real transcript
6. Cleanup (stop the bridge)


## 1. Install the two projects (editable, from local source)

`pip install -e <path>` installs directly from the local directory — this repo has never
been published to PyPI, and none of these commands need it to be. `[dev,examples]` are
optional-dependency groups defined in each project's own `pyproject.toml`, not PyPI
packages. See [`docs/LANGCHAIN.md`](../docs/LANGCHAIN.md) if this looks surprising.


In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "docs" / "ARCHITECTURE.md").exists():
    if REPO_ROOT.parent == REPO_ROOT:
        raise RuntimeError(
            "Could not locate the meetstream-langchain repo root. "
            "Run this notebook from inside the cloned repository."
        )
    REPO_ROOT = REPO_ROOT.parent

sys.path.insert(0, str(REPO_ROOT / "scripts" / "verification"))
import _lib

print("Repo root:", REPO_ROOT)
print("Python:", sys.executable)


In [ ]:
!{sys.executable} -m pip install -e "{REPO_ROOT / 'bridge'}[dev]"
!{sys.executable} -m pip install -e "{REPO_ROOT}[dev,examples]"


In [ ]:
# Editable installs register their import hooks via a .pth file that Python's
# `site` module only processes at interpreter startup -- a package installed
# mid-session (like the one above) isn't importable in *this* kernel without
# either a restart or pointing sys.path directly at its source directory, which
# is what this cell does. The bridge subprocess started later doesn't hit this
# -- it's a fresh process that reads the now-complete install normally.
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


## 2. Configuration — insert your real credentials here

Replace every `<INSERT ... HERE>` below. Nothing here is committed anywhere — this cell only
sets local Python variables for the rest of this notebook.

- `MEETSTREAM_API_KEY` — **required**. Your real MeetStream key; the bridge sends this to
  MeetStream as `Authorization: Token <key>`.
- `MEETING_URL` — a real, currently-joinable Google Meet / Zoom / Teams URL. Use this to have
  the bot dispatched by this notebook join a live meeting.
- `EXISTING_BOT_ID` — alternative to `MEETING_URL`: a `bot_id` from a bot you already
  dispatched (via this notebook or `live_meetstream_test.py dispatch`) whose transcript has
  already finished processing. Fill in exactly one of `MEETING_URL` / `EXISTING_BOT_ID`.
- `OPENAI_API_KEY` — needed only for the agent and RAG cells (real LLM calls). Leave as the
  placeholder to skip just those cells; everything else still runs for real.


In [ ]:
MEETSTREAM_API_KEY = "ms_Au2orMJHat4tuWCgFqjiPPh5VhzPaNzN"

# Fill in exactly one of these two:
MEETING_URL = "https://meet.google.com/jsa-qvvo-rby"
EXISTING_BOT_ID = ""  # e.g. "bot_123" -- leave blank if using MEETING_URL instead

# Only needed for the agent/RAG cells -- leave as the placeholder to skip just those.
OPENAI_API_KEY = "<INSERT YOUR REAL OPENAI API KEY HERE>"


def _is_placeholder(value: str) -> bool:
    return not value or value.startswith("<INSERT")


if _is_placeholder(MEETSTREAM_API_KEY):
    raise ValueError("Set MEETSTREAM_API_KEY above to your real MeetStream API key before continuing.")

if _is_placeholder(MEETING_URL) and not EXISTING_BOT_ID:
    raise ValueError("Set MEETING_URL to a real meeting URL, or EXISTING_BOT_ID to an already-dispatched bot_id.")
if not _is_placeholder(MEETING_URL) and EXISTING_BOT_ID:
    raise ValueError("Set only one of MEETING_URL / EXISTING_BOT_ID, not both.")
if _is_placeholder(MEETING_URL):
    MEETING_URL = ""  # normalize the unused placeholder to empty

RUN_LLM_CELLS = not _is_placeholder(OPENAI_API_KEY)
if RUN_LLM_CELLS:
    import os

    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY  # langchain_openai reads this from the environment
else:
    print("OPENAI_API_KEY left as placeholder -- agent and RAG cells below will be skipped.")


## 3. Start the bridge

Reuses `TemporaryBridge` from `scripts/verification/_lib.py` — the same helper the
verification suite uses to spin up a real `uvicorn app.main:app` subprocess and wait for
`/health`, pointed at your real `MEETSTREAM_API_KEY`. This calls `__enter__`/`__exit__`
manually (instead of a `with` block) so the bridge stays alive across the rest of this
notebook; it's stopped explicitly in the Cleanup section at the end.


In [ ]:
bridge = _lib.TemporaryBridge(api_key=MEETSTREAM_API_KEY)
bridge.__enter__()
print("Bridge running at", bridge.base_url)
print("Health check:", _lib.bridge_is_running(bridge.base_url))


## 4. LangChain: client + tools


In [ ]:
from langchain_meetstream import MeetStreamAPIError, MeetStreamClient, MeetStreamLoader
from langchain_meetstream.tools import get_meetstream_tools

lc_client = MeetStreamClient(api_key=MEETSTREAM_API_KEY, base_url=bridge.base_url)
lc_tools = get_meetstream_tools(lc_client)
print("LangChain tools:", [t.name for t in lc_tools])


### Dispatch a bot, or reuse an existing `bot_id`


In [ ]:
if MEETING_URL:
    dispatch_result = lc_client.dispatch_bot(meeting_url=MEETING_URL, bot_name="Notebook Test Bot")
    bot_id = dispatch_result.meeting_id
    print("Dispatched:", dispatch_result)
else:
    bot_id = EXISTING_BOT_ID
    print("Using existing bot_id:", bot_id)


### Check meeting status


In [ ]:
print(lc_client.get_meeting(bot_id))


### Wait for and fetch the transcript

Transcription is asynchronous on MeetStream's side -- this polls `get_transcript`, treating
`transcript_not_ready` as "try again shortly" rather than a failure, same as
`live_langchain_test.py`. If you just dispatched a fresh bot, join the meeting and let it run
a bit before executing this cell, or it will spend its retries waiting.


In [ ]:
import time


def wait_for_transcript(client, meeting_id, attempts=10, delay_seconds=15):
    for attempt in range(attempts):
        try:
            return client.get_transcript(meeting_id)
        except MeetStreamAPIError as exc:
            if exc.code != "transcript_not_ready":
                raise
            print(f"Not ready yet ({attempt + 1}/{attempts}) -- waiting {delay_seconds}s...")
            time.sleep(delay_seconds)
    raise TimeoutError(f"Transcript for {meeting_id} still not ready after {attempts} attempts")


transcript = wait_for_transcript(lc_client, bot_id)
print(f"{len(transcript.segments)} real segment(s)")


### Load as LangChain `Document`s


In [ ]:
lc_docs = MeetStreamLoader(meeting_id=bot_id, client=lc_client).load()
for doc in lc_docs[:3]:
    print(doc.page_content)
    print(doc.metadata)
    print()


### Invoke a tool directly (no LLM)

Proves the tool itself works, independent of whether an LLM would have chosen to call it --
same idea as `live_langchain_tools.py`.


In [ ]:
transcript_tool = next(t for t in lc_tools if t.name == "get_meeting_transcript")
print(transcript_tool.invoke({"meeting_id": bot_id}))


### Run a real LangChain agent

Same check as `scripts/verification/live_langchain_agent.py`: does a real tool-calling model
decide on its own to call `get_meeting_transcript`. Skipped if `OPENAI_API_KEY` was left as
the placeholder in section 2.


In [ ]:
if RUN_LLM_CELLS:
    from langchain.agents import create_agent
    from langchain_openai import ChatOpenAI

    lc_agent = create_agent(model=ChatOpenAI(model="gpt-4o-mini"), tools=lc_tools)
    lc_result = lc_agent.invoke(
        {"messages": [{"role": "user", "content": f"What did people say in meeting {bot_id}? Give me a short summary."}]}
    )
    print(lc_result["messages"][-1].content)
else:
    print("Skipping -- OPENAI_API_KEY not set in section 2.")


## 5. RAG over the real transcript

Minimal inline version of `examples/rag_example.py` — reuses the `Document`s already loaded
above rather than re-fetching. Skipped if `OPENAI_API_KEY` was left as the placeholder in
section 2.


In [ ]:
if RUN_LLM_CELLS:
    from langchain_core.vectorstores import InMemoryVectorStore
    from langchain_openai import ChatOpenAI, OpenAIEmbeddings
    from langchain_text_splitters import RecursiveCharacterTextSplitter

    chunks = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50).split_documents(lc_docs)
    store = InMemoryVectorStore.from_documents(chunks, OpenAIEmbeddings())

    question = "What was discussed in this meeting?"
    context = "\n\n".join(d.page_content for d in store.similarity_search(question, k=4))
    answer = ChatOpenAI(model="gpt-4o-mini").invoke(f"Answer using only this context:\n{context}\n\nQuestion: {question}")
    print(answer.content)
else:
    print("Skipping -- OPENAI_API_KEY not set in section 2.")


## 6. Cleanup


In [ ]:
bridge.__exit__(None, None, None)
print("Bridge stopped.")
